# Lab 1B: Transform Bronze into Silver

Run **Lab 1A** first. Attach `seer_livelabs_spark` and run these cells in order. Each task reads saved upstream tables, performs one complete business transformation and recreates one Silver Delta table. No separate validation task or reference answer key is used. Source exceptions stay visible in the transformed data.

After this notebook, run **Lab 1C: Build Gold**.

## Rerunning Lab 1B

No manual deletion is required. Each transformation drops and recreates its named Silver table without a backup; Bronze and unrelated objects remain unchanged. A failed write after a drop may leave that target absent.

If source data changed, run Lab 1A first. Otherwise, with current Bronze inputs, start at Silver setup and execute all transformations in order because some Silver tables depend on earlier Silver outputs. Continue **Lab 1C → Lab 2** afterward; Gold tables and predictions do not refresh automatically. Resolve failed writes before downstream work. Avoid concurrent writers, then refresh the catalog and verify counts and lineage.

Do not delete experiments, model versions or the KB to rerun Silver. See `docs/reruns.html` in the downloaded workshop for both labs' rerun and cleanup guidance.


In [ ]:
import json
from pathlib import Path
from pyspark.sql import functions as F

config = json.loads(Path("/Workspace/Shared/seer-aidp-livelabs/config/workshop.json").read_text())
BRONZE = f"{config['catalog']}.{config['schemas']['bronze']}"
SILVER = f"{config['catalog']}.{config['schemas']['silver']}"

# Create the silver schema if it does not already exist.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")


## Task 1: Standardize and reconcile suppliers


In [ ]:
# Normalize supplier names, source status codes, certifications and location spelling.
source_suppliers = spark.table(f"{BRONZE}.suppliers_raw")
name = F.trim(F.col("supplier_name"))
status = F.upper(F.trim(F.col("source_status")))
cert = F.upper(F.trim(F.coalesce(F.col("certification"), F.lit(""))))
standardized = source_suppliers.select(
    "source_record_id", "source_system", "ingestion_batch_id",
    F.when(F.upper(name).isin("ATLAS STRUCTURAL FAB.", "ATLAS STRUCTURAL FABRICATION"),
           "Atlas Structural Fabrication").otherwise(F.initcap(name)).alias("canonical_supplier_name"),
    F.when(status.isin("A", "APPROVED"), "APPROVED")
     .when(status == "PENDING_INFO", "REQUEST_INFORMATION")
     .otherwise("REVIEW_REQUIRED").alias("qualification_status"),
    F.when(cert == "", "MISSING").when(cert.contains("AISC"), "AISC")
     .otherwise(cert).alias("normalized_certification"),
    F.regexp_replace(F.upper(F.trim("location")), ", TEXAS", ", TX").alias("normalized_location"))

# Reconcile duplicate source records into one supplier; retain the originals for traceability.
suppliers = standardized.groupBy("canonical_supplier_name").agg(
    F.collect_set("qualification_status").alias("statuses"),
    F.collect_list(F.struct("*")).alias("source_records"),
    F.max((F.col("normalized_certification") == "MISSING").cast("int")).alias("missing_cert"),
    F.count("*").alias("matched_source_count"),
    F.countDistinct("source_system").alias("source_system_count"),
    F.to_json(F.sort_array(F.collect_list("source_record_id"))).alias("source_record_ids"),
    F.to_json(F.sort_array(F.collect_set("normalized_certification"))).alias("certifications"))

# Build a stable supplier ID and retain the most cautious qualification and compliance result.
suppliers = suppliers.select(
    F.concat(F.lit("SUP-"), F.upper(F.substring(F.sha2("canonical_supplier_name", 256), 1, 12))).alias("supplier_id"),
    "canonical_supplier_name",
    F.when(F.array_contains("statuses", "REVIEW_REQUIRED"), "REVIEW_REQUIRED")
     .when(F.array_contains("statuses", "REQUEST_INFORMATION"), "REQUEST_INFORMATION")
     .otherwise("APPROVED").alias("qualification_status"),
    F.when(F.col("missing_cert") == 1, "REVIEW_REQUIRED").otherwise("EVIDENCE_PRESENT").alias("compliance_status"),
    "matched_source_count", "source_system_count",
    F.when(F.size("statuses") > 1, "CONFLICT_RETAINED").otherwise("MATCHED").alias("reconciliation_status"),
    "source_record_ids", "certifications", "source_records")
# Recreate and save this complete transformation once to preserve table-to-table lineage.
spark.sql(f"DROP TABLE IF EXISTS {SILVER}.suppliers")
suppliers.write.format("delta").mode("overwrite").saveAsTable(f"{SILVER}.suppliers")
display(spark.table(f"{SILVER}.suppliers"))


## Task 2: Reconcile assets with project names


In [ ]:
# Build a project-name lookup from schedules and assign the explicit project-to-asset mapping.
projects = spark.table(f"{BRONZE}.schedules_raw").select(
    F.col("project_reference").alias("project_id"), "project_name").distinct()
assets = (spark.table(f"{BRONZE}.assets_raw")
    .withColumn("asset_id", (F.when(F.col("project_reference") == "AUS-BANK-01", "STR-AUS-STEEL-01")
        .when(F.col("project_reference") == "HOU-MIXED-02", "LAB-HOU-STEEL-02")
        .when(F.col("project_reference") == "HAR-SEISMIC-03", "LAB-HAR-BRACE-03")))
    .withColumnRenamed("project_reference", "project_id")
    .withColumnRenamed("asset_name", "canonical_asset_name")
    .withColumnRenamed("asset_category", "asset_type")
    .withColumnRenamed("financial_status", "normalized_status"))
# Add project names without dropping assets that have no matching schedule.
assets = assets.join(projects, "project_id", "left").withColumn(
    "reconciliation_status", F.lit("EXPLICIT_FIXTURE_CROSSWALK"))
# Recreate and save this complete transformation once to preserve table-to-table lineage.
spark.sql(f"DROP TABLE IF EXISTS {SILVER}.assets")
assets.write.format("delta").mode("overwrite").saveAsTable(f"{SILVER}.assets")
display(spark.table(f"{SILVER}.assets"))


## Task 3: Resolve purchase orders and convert amounts


In [ ]:
# Resolve asset IDs by project and supplier IDs by their original source identifiers.
asset_keys = spark.table(f"{SILVER}.assets").select(
    F.col("project_id").alias("project_reference"), "asset_id")
supplier_keys = spark.table(f"{SILVER}.suppliers").select(
    "supplier_id", F.explode("source_records").alias("source")).select(
    "supplier_id", F.col("source.source_record_id").alias("supplier_reference"))
# Left joins retain unresolved suppliers. Convert decimal currency to integer cents.
po = (spark.table(f"{BRONZE}.purchasing_raw").join(asset_keys, "project_reference", "left")
    .join(supplier_keys, "supplier_reference", "left")
    .withColumn("project_id", F.col("project_reference"))
    .withColumn("po_status", F.upper(F.trim("po_status")))
    .withColumn("amount_cents", (F.col("amount").cast("decimal(20,2)") * 100).cast("long")))
# Recreate and save this complete transformation once to preserve table-to-table lineage.
spark.sql(f"DROP TABLE IF EXISTS {SILVER}.purchase_orders")
po.write.format("delta").mode("overwrite").saveAsTable(f"{SILVER}.purchase_orders")
display(spark.table(f"{SILVER}.purchase_orders"))


## Task 4: Normalize inspection results


In [ ]:
# Keep every inspection with a left join, add the canonical asset ID, and normalize status text.
inspections = (spark.table(f"{BRONZE}.inspections_raw").join(
    spark.table(f"{SILVER}.assets").select(F.col("project_id").alias("project_reference"), "asset_id"),
    "project_reference", "left").withColumn("project_id", F.col("project_reference"))
    .withColumn("result_status", F.upper(F.trim("result_status"))))
# Recreate and save this complete transformation once to preserve table-to-table lineage.
spark.sql(f"DROP TABLE IF EXISTS {SILVER}.inspections")
inspections.write.format("delta").mode("overwrite").saveAsTable(f"{SILVER}.inspections")

# Preview up to five records from the saved table; row order is not guaranteed.
print(f"Preview of {SILVER}.inspections (up to 5 records):")
spark.table(f"{SILVER}.inspections").show(5, truncate=False)


## Task 5: Normalize project milestones


In [ ]:
# Keep every scheduled activity, resolve its asset, and normalize its status without changing dates.
milestones = (spark.table(f"{BRONZE}.schedules_raw").join(
    spark.table(f"{SILVER}.assets").select(F.col("project_id").alias("project_reference"), "asset_id"),
    "project_reference", "left").withColumn("project_id", F.col("project_reference"))
    .withColumn("milestone_status", F.upper(F.trim("milestone_status"))))
# Recreate and save this complete transformation once to preserve table-to-table lineage.
spark.sql(f"DROP TABLE IF EXISTS {SILVER}.milestones")
milestones.write.format("delta").mode("overwrite").saveAsTable(f"{SILVER}.milestones")

# Preview up to five records from the saved table; row order is not guaranteed.
print(f"Preview of {SILVER}.milestones (up to 5 records):")
spark.table(f"{SILVER}.milestones").show(5, truncate=False)


### Review the Silver tables and row counts

Now we can see all the tables in the Silver schema and the number of rows in each table. Run the next cell to check that the expected tables were created and contain data after running the transformations. If a table is missing or its count is unexpected, review the earlier cells and their outputs.

This is a quick loading check, not a full data-quality validation. The cell only reads the tables; it does not change any data.


In [ ]:
# Print each saved Silver table and its current record count (read-only).
print(f"{'Silver table':<80} {'Record count':>12}")
for table_name in sorted(table.name for table in spark.catalog.listTables(SILVER) if not table.isTemporary):
    full_name = f"{SILVER}.{table_name}"
    record_count = spark.table(full_name).count()
    print(f"{full_name:<80} {record_count:>12,}")
